# Arm verdicts for the TM border-band and floor-clip run: 49 ROIs x 6 clicks, `mo_pad61`, post-D11

**What this is.** A read-only consolidation of `tm_border_floor_49roi_6seed.ipynb`, written after its audit
(`Research Logs/2026-10-01-tm-border-floor-49roi-6seed-audit.md`). No search is re-run. Every number comes from that
run's CSVs and its search pickle, with the run's own statistics functions copied verbatim (asserted below). It is
**post hoc**: nothing here was pre-registered beyond what the source notebook pre-registered, and every number is
in-sample on the 49 ROIs of `images/extra_valid`, which D5 calls an optimistic draw.

**Why it exists.** The audit reproduced every number and the "recommend `A3`" decision, but corrected three readings:
- **Tier 1.** "The clip is not a targeted removal of white-adjacent false positives" rested on a churn count that
  treats a 1-pixel peak shift as a different candidate.
- **Tier 2.** The pool and chromatin-od flows used the same exact-coordinate count, and "the extra pool mitoses do not
  reach od's top 10" is wrong.
- **Tier 2.** The +3.71 headline bundles the band fix, which by the run's own premise is a benchmark-only correction.

This notebook redoes those on the contrasts that matter and gives one verdict per arm.

**Two premises, fixed from outside the data.**
- **The product shows the whole slide with zoom**, so a template never reads a replicate pad: the ROI border is a
  benchmark artefact (user, 2026-09-30). Fix 1 (band out) is therefore a correction to the benchmark harness, and the
  product estimate of the clip is **`A3 - A1`**, the clip with the harness corrected on both sides. `A2 - A0`, the clip
  on the uncorrected harness, is shown beside it.
- **The production ranker is `tm_score`** (D5). `chromatin_od` is a reported second axis.

**The arms** (from the source notebook).

| arm | border band | floor | role in the run |
|---|---|---|---|
| `A0` | production (replicate pad, band valid) | none | the recorded `mo_pad61` search |
| `A1` | invalid and floored | none | fix 1 alone |
| `A1v` | invalid only (the prompt's spec) | none | descriptive |
| `A2` | production | `b_otsu` | fix 2 alone |
| `A3` | invalid and floored | `b_otsu` | both; the pre-registered candidate |
| `A3v` | invalid only | `b_otsu` | descriptive |
| `A2m` | production | `b_med` (ROI median) | floor sensitivity |

**Inputs.** `results/tm_border_floor_49roi_6seed_{per_run,top30,delta_stats,white,floors}.csv`; the search pickle,
copied from `$TMPDIR` to `results/tm_border_floor_49roi_6seed_search.pkl` on 2026-10-01 (SHA-256 asserted); the
audit's two churn tables, used only as a cross-check; `databases/MIDOG++.json` for re-matching the pools.

**Outputs.** `results/tm_border_floor_arm_verdicts_*.csv`. The notebook refuses to overwrite them.

In [1]:
import hashlib
import inspect
import itertools
import json
import math
import os
import pickle
import sys
from fractions import Fraction

import numpy as np
import pandas as pd
from scipy import stats as sps

sys.path.insert(0, '../..')
from midog_utils import dataset as ds
from midog_utils import evaluate as ev

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 400)
pd.set_option('display.max_colwidth', 140)

# the run this notebook judges; the search pickle was copied out of $TMPDIR on 2026-10-01, where it was the only home of the full pools
SRC_NB = 'tm_border_floor_49roi_6seed.ipynb'
RUN = '../../results/tm_border_floor_49roi_6seed'
AUDIT = '../../results/tm_border_floor_49roi_6seed_audit'
FG_SRC = '../../results/precision_at_k_49roi_6seed_fillgeom_postD11'
DB_PATH = '../../databases/MIDOG++.json'
INPUTS = {'per_run': f'{RUN}_per_run.csv', 'top30': f'{RUN}_top30.csv', 'delta_stats': f'{RUN}_delta_stats.csv', 'white': f'{RUN}_white.csv', 'floors': f'{RUN}_floors.csv', 'search_pickle': f'{RUN}_search.pkl', 'audit_churn_top10': f'{AUDIT}_churn_top10.csv', 'audit_churn_pool': f'{AUDIT}_churn_pool.csv', 'fillgeom_per_run': f'{FG_SRC}_per_run.csv', 'midog_json': DB_PATH}
PICKLE_SHA256 = '0f921c65acaa1164b8475d8c266b936123a0718a07a4d0f010162cca8cec7d69'   # the $TMPDIR original, hashed when it was copied
OUT_STEM = '../../results/tm_border_floor_arm_verdicts'
OUT = {name: f'{OUT_STEM}_{name}.csv' for name in ('contrasts', 'white_checks', 'bounds', 'churn_top10', 'white_dose', 'white_dose_gap', 'pool_od_flows', 'domain', 'domain_level', 'per_click', 'robustness', 'verdicts')}
assert not set(map(os.path.abspath, OUT.values())) & set(map(os.path.abspath, INPUTS.values())), 'an output path would overwrite an input'
existing = [p for p in OUT.values() if os.path.exists(p)]
assert not existing, f'this notebook never overwrites its outputs; these already exist: {existing}'

# copied from cell 1 of the source notebook: the statistics and the contrast plan read them
SEED_INDICES = (0, 1, 2, 3, 4, 5)
N_CLICKS_PER_ROI = len(SEED_INDICES)
N_BOOT = 10_000
BOOT_SEED = 20260917
AXES = ('chromatin_od', 'tm_score')
BUDGETS = (10, 20, 30)
ALPHA = 0.05
WHITE_SHARE = 0.20             # "white-adjacent": more than this share of the 61 px window is white (gray > 215)
TM_ARMS = {'A0': ('production', None), 'A1': ('floored', None), 'A1v': ('invalid', None), 'A2': ('production', 'b_otsu'), 'A3': ('floored', 'b_otsu'), 'A3v': ('invalid', 'b_otsu'), 'A2m': ('production', 'b_med')}
ARM_NAMES = tuple(TM_ARMS)
MAIN_ARMS = ('A0', 'A1', 'A2', 'A3', 'A2m')
CONTRASTS = {'A3 - A0': ('A3', 'A0'), 'A1 - A0': ('A1', 'A0'), 'A2 - A0': ('A2', 'A0'), 'A3 - A1': ('A3', 'A1'), 'A3 - A2': ('A3', 'A2'), 'A2m - A2': ('A2m', 'A2'), 'A1 - A1v': ('A1', 'A1v'), 'A3 - A3v': ('A3', 'A3v'), 'A2m - A0': ('A2m', 'A0')}
FAMILIES = {
    'primary': [('tm_score', 'A3 - A0')],
    'tm_decomposition': [('tm_score', c) for c in ('A1 - A0', 'A2 - A0', 'A3 - A1', 'A3 - A2')],
    'od_five': [('chromatin_od', c) for c in ('A3 - A0', 'A1 - A0', 'A2 - A0', 'A3 - A1', 'A3 - A2')],
    'median_floor': [(axis, 'A2m - A2') for axis in ('tm_score', 'chromatin_od')],
}
DESCRIPTIVE = [(axis, c) for c in ('A1 - A1v', 'A3 - A3v', 'A2m - A0') for axis in ('tm_score', 'chromatin_od')]
FAMILY_SIZE = {'primary': 3, 'tm_decomposition': 12, 'od_five': 15, 'median_floor': 6}

# this notebook
TOLS = (0, 1, 3)                  # px, Chebyshev: a top-10 candidate is the same object if the other list has one this close; 0 is the source notebook's exact-coordinate count
WHITE_CUTS = (0.10, 0.20, 0.30)   # the white-adjacency cut-off, varied; 0.20 is the pre-registered one
CLIP_PAIRS = (('A1', 'A3'), ('A0', 'A2'))   # the clip with the harness corrected on both sides (the product estimate), and the clip on the uncorrected harness
FLOW_ARMS = ('A0', 'A1', 'A2', 'A3')
PRODUCT_AXIS = 'tm_score'         # D5: the production ranker is the TM_CCOEFF score; chromatin od is a reported second axis
TP = ev.HUMAN_CORRECT_LABEL


def sha256_12(path):
    """
        The first 12 hex digits of a file's SHA-256, so the inputs this notebook read are on record.

        path (str): the file to hash.

        Returns str: 12 hex characters.
    """
    with open(path, 'rb') as fh:
        return hashlib.sha256(fh.read()).hexdigest()[:12]


INPUT_SHA = {name: sha256_12(p) for name, p in INPUTS.items()}
for name, p in INPUTS.items():
    print(f'  {INPUT_SHA[name]}  {p}')
assert INPUT_SHA['search_pickle'] == PICKLE_SHA256[:12], 'the search pickle is not the copy of the run that wrote the CSVs'


  ef7940000fec  ../../results/tm_border_floor_49roi_6seed_per_run.csv
  4068d04b7886  ../../results/tm_border_floor_49roi_6seed_top30.csv
  e87350f523e7  ../../results/tm_border_floor_49roi_6seed_delta_stats.csv
  95c48e3df9d6  ../../results/tm_border_floor_49roi_6seed_white.csv
  a8e83aa2692b  ../../results/tm_border_floor_49roi_6seed_floors.csv
  0f921c65acaa  ../../results/tm_border_floor_49roi_6seed_search.pkl
  a9a4d9516dd7  ../../results/tm_border_floor_49roi_6seed_audit_churn_top10.csv
  1714b0fe081b  ../../results/tm_border_floor_49roi_6seed_audit_churn_pool.csv
  8668d2697420  ../../results/precision_at_k_49roi_6seed_fillgeom_postD11_per_run.csv
  8c87ab4276eb  ../../databases/MIDOG++.json


## Gates

Before any new number:
1. the pickle is the run that wrote the CSVs (every `per_run` cell and every `top30` row);
2. the statistics functions are verbatim copies of the source notebook's (cells 10, 11, 21 and 23);
3. those functions reproduce all 54 recorded `delta_stats` rows, every column, and all 20 recorded M2/M3 rows.

In [2]:
PER_RUN = pd.read_csv(INPUTS['per_run'], float_precision='round_trip')
TOP = pd.read_csv(INPUTS['top30'], float_precision='round_trip')
REC_DELTA = pd.read_csv(INPUTS['delta_stats'], float_precision='round_trip')
REC_WHITE = pd.read_csv(INPUTS['white'], float_precision='round_trip')
FLOORS = pd.read_csv(INPUTS['floors'], float_precision='round_trip')
with open(INPUTS['search_pickle'], 'rb') as fh:
    PK = pickle.load(fh)
POOL = PK['POOL']
assert (len(PER_RUN), len(TOP), len(REC_DELTA), len(REC_WHITE), len(FLOORS), len(POOL)) == (4116, 123480, 54, 20, 49, 203441), 'an input does not have its expected row count'
assert set(PER_RUN['arm']) == set(ARM_NAMES) and not PER_RUN.duplicated(['file_name', 'seed_index', 'arm', 'rank_key']).any()

# the pickle is the run that wrote the CSVs: PER_RUN cell for cell, and the top 30 lists row for row
pk_run = PK['PER_RUN'].reset_index(drop=True)
assert list(pk_run.columns) == list(PER_RUN.columns)
n_cells = 0
for c in PER_RUN.columns:
    a, b = pk_run[c], PER_RUN[c]
    if pd.api.types.is_numeric_dtype(b) and not pd.api.types.is_bool_dtype(b):
        ok = np.isclose(pd.to_numeric(a, errors='coerce').to_numpy(float), b.to_numpy(float), rtol=1e-12, atol=0, equal_nan=True)
    else:
        ok = (a.astype(str).replace('None', 'nan') == b.astype(str)).to_numpy()
    assert ok.all(), f'pickle PER_RUN differs from per_run.csv in {c}'
    n_cells += len(ok)
pk_top = PK['TOP'].reset_index(drop=True)
for c in ('rank', 'cx', 'cy', 'matched_ann_id', 'seed_index'):
    assert np.array_equal(pk_top[c].to_numpy(), TOP[c].to_numpy()), f'pickle TOP differs from top30.csv in {c}'
for c in ('bucket', 'file_name', 'arm', 'rank_key'):
    assert (pk_top[c] == TOP[c]).all(), f'pickle TOP differs from top30.csv in {c}'
assert np.allclose(pk_top['white_frac_61'], TOP['white_frac_61'], rtol=1e-12, atol=0)
print(f'pickle == per_run.csv on all {n_cells} cells and == top30.csv on all {len(TOP)} rows')

ROI_ORDER = list(pd.unique(pd.read_csv(INPUTS['fillgeom_per_run'], usecols=['file_name'])['file_name']))   # the bootstrap's ROI order, as in the source notebook
N_ROIS = len(ROI_ORDER)
assert N_ROIS == 49 and set(ROI_ORDER) == set(PER_RUN['file_name'])
DOMAIN_OF = PER_RUN.drop_duplicates('file_name').set_index('file_name')['domain']
DOMAINS = list(pd.unique(PER_RUN['domain']))
assert len(DOMAINS) == 7 and DOMAIN_OF.value_counts().eq(7).all(), '7 ROIs in each of 7 domains'
SHORT = {'canine cutaneous mast cell tumor': 'mast cell', 'canine lung cancer': 'lung', 'canine lymphosarcoma': 'lymphosarcoma', 'canine soft tissue sarcoma': 'soft tissue sarcoma', 'human breast cancer': 'breast', 'human melanoma': 'melanoma', 'human neuroendocrine tumor': 'neuroendocrine'}
assert set(SHORT) == set(DOMAINS)


pickle == per_run.csv on all 222264 cells and == top30.csv on all 123480 rows


In [3]:
def sign_flip_null_counts(abs_deltas):
    """
        Exact null distribution of sum(+/-|d_i|) by convolution over integer offsets.

        abs_deltas (array-like): nonzero absolute integer deltas, one per ROI.

        Returns np.ndarray: int64 counts of each sum -T..T (index = sum + T, T = sum |d_i|), totalling 2 ** n.
    """
    d = [int(v) for v in abs_deltas]
    assert all(v > 0 for v in d) and len(d) <= 62, 'nonzero integer deltas only, and 2 ** n must fit int64'
    total = sum(d)
    counts = np.zeros(2 * total + 1, dtype=np.int64)
    counts[total] = 1
    for v in d:
        shifted = np.zeros_like(counts)
        shifted[v:] += counts[:len(counts) - v]
        shifted[:len(counts) - v] += counts[v:]
        counts = shifted
    return counts


def exact_sign_flip_p(deltas):
    """
        Two-sided exact sign-flip p on integer per-ROI deltas, by dynamic-programming convolution.

        deltas (array-like): one integer delta per ROI; zeros carry no sign.

        Returns tuple[float, int]: the p-value and the number of nonzero deltas.
    """
    d = np.asarray(deltas)
    assert np.all(d == np.round(d)), 'the convolution needs integer deltas'
    d = d.astype(np.int64)
    nz = np.abs(d[d != 0])
    if len(nz) == 0:
        return 1.0, 0
    counts = sign_flip_null_counts(nz)
    total = int(nz.sum())
    observed = abs(int(d.sum()))
    sums = np.arange(-total, total + 1)
    return float(Fraction(int(counts[np.abs(sums) >= observed].sum()), 2 ** len(nz))), len(nz)


def holm_adjust(pvalues):
    """
        Holm step-down adjusted p-values, valid under any dependence between the tests.

        pvalues (array-like): raw p-values of one family of tests.

        Returns np.ndarray: adjusted p-values, in the input order.
    """
    p = np.asarray(pvalues, dtype=float)
    order = np.argsort(p, kind='stable')
    adjusted = np.maximum.accumulate((len(p) - np.arange(len(p))) * p[order]).clip(max=1.0)
    out = np.empty_like(adjusted)
    out[order] = adjusted
    return out


def boot_index(n_roi, stream=0):
    """
        The cluster-bootstrap resample of ROI indices, seeded exactly as every earlier run seeds it.

        n_roi (int): the number of ROIs being resampled.
        stream (int): 0 for the reported CI; 1.. for the re-seeding stability check.

        Returns np.ndarray: (N_BOOT, n_roi) resampled ROI indices.
    """
    return np.random.default_rng([BOOT_SEED, stream]).integers(n_roi, size=(N_BOOT, n_roi))


def boot_ci_exact(int_values, idx, denominator):
    """
        Percentile 95% CI of a pooled ratio over resampled ROIs, from exact integer numerators.

        int_values (np.ndarray): one integer numerator per ROI.
        idx (np.ndarray): (n_boot, n_roi) resampled ROI indices.
        denominator (int): the pooled denominator per resample.

        Returns tuple[float, float, float, float]: CI low, CI high, and the share of resamples at or above 0 and at or below 0.
    """
    sums = np.asarray(int_values, dtype=np.int64)[idx].sum(axis=1)
    lo, hi = np.percentile(sums / denominator, [2.5, 97.5])
    return float(lo), float(hi), float((sums >= 0).mean()), float((sums <= 0).mean())


def pair_stats(d, k, idx):
    """
        Every statistic for one contrast at one budget, from its per-ROI click-summed integer deltas.

        d (np.ndarray): one integer delta per ROI, in ROI order.
        k (int): the budget K the deltas count true positives in.
        idx (np.ndarray): (N_BOOT, len(d)) bootstrap ROI indices.

        Returns dict: one delta_stats row without its Holm columns.
    """
    d = np.asarray(d, dtype=np.int64)
    n_roi = len(d)
    denominator = N_CLICKS_PER_ROI * n_roi * k
    pooled = d.sum() / denominator
    wins, losses, ties = int((d > 0).sum()), int((d < 0).sum()), int((d == 0).sum())
    exact_p, n_nonzero = exact_sign_flip_p(d)
    ci_low, ci_high, share_ge_0, share_le_0 = boot_ci_exact(d, idx, denominator)
    x = d / (N_CLICKS_PER_ROI * k)
    half = sps.t.ppf(0.975, n_roi - 1) * x.std(ddof=1) / math.sqrt(n_roi) if n_roi > 1 else np.nan
    t_ci_low, t_ci_high = float(x.mean() - half), float(x.mean() + half)
    ci_excludes_0 = bool(ci_low > 0 or ci_high < 0)
    n_same_direction = wins if pooled > 0 else (losses if pooled < 0 else 0)
    majority = bool(n_same_direction > n_roi / 2)
    return dict(n_roi=n_roi, n_clicks=N_CLICKS_PER_ROI * n_roi, pooled_delta_precision=pooled, wins=wins, losses=losses, ties=ties, n_nonzero=n_nonzero, exact_p=exact_p, min_attainable_p=2.0 ** (1 - n_nonzero) if n_nonzero else 1.0, ci_low=ci_low, ci_high=ci_high, boot_share_ge_0=share_ge_0, boot_share_le_0=share_le_0, t_ci_low=t_ci_low, t_ci_high=t_ci_high, ci_excludes_0=ci_excludes_0, t_ci_excludes_0=bool(t_ci_low > 0 or t_ci_high < 0), n_same_direction=n_same_direction, majority_of_all_rois=majority, d5_bar_met=bool(ci_excludes_0 and majority))


def pair_stats_clicks(d, n_clicks, k, idx):
    """
        pair_stats generalised to a ROI-varying click count: the pooled delta, the bootstrap and the t interval weight each ROI by its clicks.

        d (np.ndarray): one integer delta per ROI, in ROI order, summed over that ROI's included clicks.
        n_clicks (np.ndarray): the number of included clicks per ROI, in the same order.
        k (int): the budget K the deltas count true positives in.
        idx (np.ndarray): (N_BOOT, len(d)) bootstrap ROI indices.

        Returns dict: one delta_stats row without its Holm columns, with the same keys as pair_stats.
    """
    d = np.asarray(d, dtype=np.int64)
    n = np.asarray(n_clicks, dtype=np.int64)
    assert len(d) == len(n) and (n > 0).all(), 'one positive click count per ROI'
    n_roi = len(d)
    denominator = int(n.sum()) * k
    pooled = d.sum() / denominator
    wins, losses, ties = int((d > 0).sum()), int((d < 0).sum()), int((d == 0).sum())
    exact_p, n_nonzero = exact_sign_flip_p(d)
    sums = d[idx].sum(axis=1)
    lo, hi = np.percentile(sums / (n[idx].sum(axis=1) * k), [2.5, 97.5])
    ci_low, ci_high, share_ge_0, share_le_0 = float(lo), float(hi), float((sums >= 0).mean()), float((sums <= 0).mean())
    x = d / (n * k)
    half = sps.t.ppf(0.975, n_roi - 1) * x.std(ddof=1) / math.sqrt(n_roi) if n_roi > 1 else np.nan
    t_ci_low, t_ci_high = float(x.mean() - half), float(x.mean() + half)
    ci_excludes_0 = bool(ci_low > 0 or ci_high < 0)
    n_same_direction = wins if pooled > 0 else (losses if pooled < 0 else 0)
    majority = bool(n_same_direction > n_roi / 2)
    return dict(n_roi=n_roi, n_clicks=int(n.sum()), pooled_delta_precision=pooled, wins=wins, losses=losses, ties=ties, n_nonzero=n_nonzero, exact_p=exact_p, min_attainable_p=2.0 ** (1 - n_nonzero) if n_nonzero else 1.0, ci_low=ci_low, ci_high=ci_high, boot_share_ge_0=share_ge_0, boot_share_le_0=share_le_0, t_ci_low=t_ci_low, t_ci_high=t_ci_high, ci_excludes_0=ci_excludes_0, t_ci_excludes_0=bool(t_ci_low > 0 or t_ci_high < 0), n_same_direction=n_same_direction, majority_of_all_rois=majority, d5_bar_met=bool(ci_excludes_0 and majority))


def stats_for(d, n_clicks, k):
    """
        One contrast's delta_stats: the verbatim pair_stats when every ROI keeps all 6 clicks, else pair_stats_clicks.

        d (np.ndarray): one integer delta per ROI, in ROI order.
        n_clicks (np.ndarray): the number of included clicks per ROI.
        k (int): the budget K.

        Returns dict: one delta_stats row without its Holm columns, naming the function used.
    """
    n = np.asarray(n_clicks)
    idx = boot_index(len(d))
    if (n == N_CLICKS_PER_ROI).all():
        return dict(pair_stats(d, k, idx), stats_function='pair_stats')
    return dict(pair_stats_clicks(d, n, k, idx), stats_function='pair_stats_clicks')


def roi_sum(frame, axis, arm, col):
    """
        One column summed over each ROI's clicks for one arm and axis, in ROI_ORDER.

        frame (pd.DataFrame): a per-run table.
        axis (str): the rank_key.
        arm (str): the arm.
        col (str): the column to sum.

        Returns np.ndarray: int64, one value per ROI.
    """
    g = frame[(frame['rank_key'] == axis) & (frame['arm'] == arm)].groupby('file_name')[col].sum()
    return g.reindex(ROI_ORDER).to_numpy(dtype=np.int64)


def share_diff_boot(num_a, den_a, num_b, den_b, idx):
    """
        The paired difference of two ratio-of-sums shares, with a ROI-cluster bootstrap 95% CI.

        num_a (np.ndarray): per-ROI numerators of arm a.
        den_a (np.ndarray): per-ROI denominators of arm a.
        num_b (np.ndarray): per-ROI numerators of arm b.
        den_b (np.ndarray): per-ROI denominators of arm b.
        idx (np.ndarray): (N_BOOT, n_roi) resampled ROI indices.

        Returns dict: the two shares, their difference, its CI and whether the CI lies below 0.
    """
    sa, sb = num_a.sum() / den_a.sum(), num_b.sum() / den_b.sum()
    da, db = den_a[idx].sum(axis=1), den_b[idx].sum(axis=1)
    ok = (da > 0) & (db > 0)
    diff = num_a[idx].sum(axis=1)[ok] / da[ok] - num_b[idx].sum(axis=1)[ok] / db[ok]
    lo, hi = np.percentile(diff, [2.5, 97.5])
    return dict(share_a=sa, share_b=sb, diff=sa - sb, ci_low=float(lo), ci_high=float(hi), ci_below_0=bool(hi < 0), ci_above_0=bool(lo > 0), n_boot_used=int(ok.sum()))


def count_delta(d, idx):
    """
        A per-ROI integer count delta: its total, the exact two-sided sign-flip p, and a ROI-cluster bootstrap 95% CI of the total.

        d (np.ndarray): one integer delta per ROI.
        idx (np.ndarray): (N_BOOT, n_roi) resampled ROI indices.

        Returns dict: total, per click, wins/losses/ties, exact p, CI of the total.
    """
    d = np.asarray(d, dtype=np.int64)
    p, n_nz = exact_sign_flip_p(d)
    lo, hi = np.percentile(d[idx].sum(axis=1), [2.5, 97.5])
    return dict(total=int(d.sum()), per_click=d.sum() / (N_CLICKS_PER_ROI * len(d)), wins=int((d > 0).sum()), losses=int((d < 0).sum()), ties=int((d == 0).sum()), exact_p=p, n_nonzero=n_nz, ci_low=float(lo), ci_high=float(hi))


def contrast_table(frame, label):
    """
        Every pre-registered and descriptive contrast at every K on its axis, with Holm within each family.

        frame (pd.DataFrame): a per-run table (the main one, or the self-match-dropped copy).
        label (str): which table this is, carried into a column.

        Returns pd.DataFrame: one delta_stats row per (family, axis, contrast, K).
    """
    rows = []
    n_clicks = np.full(N_ROIS, N_CLICKS_PER_ROI)
    plan = [(fam, axis, c) for fam, items in FAMILIES.items() for axis, c in items] + [('descriptive', axis, c) for axis, c in DESCRIPTIVE]
    for fam, axis, c in plan:
        a, b = CONTRASTS[c]
        for k in BUDGETS:
            d = roi_sum(frame, axis, a, f'tp_at_{k}') - roi_sum(frame, axis, b, f'tp_at_{k}')
            rows.append(dict(table=label, holm_family=fam, rank_key=axis, pair=c, K=k, **stats_for(d, n_clicks, k)))
    out = pd.DataFrame(rows)
    tested = out['holm_family'] != 'descriptive'
    out.loc[tested, 'holm_p'] = out[tested].groupby('holm_family')['exact_p'].transform(lambda p: holm_adjust(p.to_numpy()))
    out['family_size'] = out['holm_family'].map(FAMILY_SIZE)
    assert out[tested].groupby('holm_family').size().reindex(list(FAMILY_SIZE)).tolist() == list(FAMILY_SIZE.values()), 'a family does not have its pre-registered number of tests'
    out['significant_gain'] = tested & (out['holm_p'] < ALPHA) & (out['pooled_delta_precision'] > 0)
    out['significant_loss'] = tested & (out['holm_p'] < ALPHA) & (out['pooled_delta_precision'] < 0)
    return out


def def_source(cell_src, name):
    """
        One top-level function's source as written in a notebook cell, from its def line to the next top-level line.

        cell_src (str): the cell's source.
        name (str): the function name.

        Returns str: the def block, trailing blank lines removed.
    """
    lines = cell_src.split('\n')
    start = next(i for i, ln in enumerate(lines) if ln.startswith(f'def {name}('))
    end = next((i for i in range(start + 1, len(lines)) if lines[i] and not lines[i][0].isspace()), len(lines))
    return '\n'.join(lines[start:end]).rstrip()


# the copies are verbatim: each function's source equals its def block in the source notebook
SRC_CELLS = [''.join(c['source']) for c in json.load(open(SRC_NB))['cells']]
COPIED = {10: ('sign_flip_null_counts', 'exact_sign_flip_p', 'holm_adjust', 'boot_index', 'boot_ci_exact', 'pair_stats'), 11: ('pair_stats_clicks', 'stats_for'), 21: ('roi_sum', 'share_diff_boot', 'count_delta'), 23: ('contrast_table',)}
for cell, names in COPIED.items():
    for name in names:
        assert inspect.getsource(globals()[name]).rstrip() == def_source(SRC_CELLS[cell], name), f'{name} is not a verbatim copy of cell {cell} of {SRC_NB}'
print(f"verbatim: {sum(len(v) for v in COPIED.values())} functions equal their def blocks in cells {', '.join(map(str, COPIED))} of {SRC_NB}")


verbatim: 12 functions equal their def blocks in cells 10, 11, 21, 23 of tm_border_floor_49roi_6seed.ipynb


In [4]:
# gate: the 54 recorded delta_stats rows, recomputed from per_run.csv with the copied functions, every column
DELTA = contrast_table(PER_RUN, 'main')
assert len(DELTA) == len(REC_DELTA) == 54
key = ['holm_family', 'rank_key', 'pair', 'K']
assert (DELTA[key].to_numpy() == REC_DELTA[key].to_numpy()).all(), 'the contrast plan differs from the recorded one'
n_checked = 0
for c in REC_DELTA.columns:
    a, b = DELTA[c], REC_DELTA[c]
    if pd.api.types.is_float_dtype(b):
        ok = np.isclose(a.to_numpy(float), b.to_numpy(float), rtol=1e-12, atol=0, equal_nan=True)
    else:
        ok = (a.to_numpy() == b.to_numpy())
    assert ok.all(), f'delta_stats column {c} does not reproduce'
    n_checked += len(ok)
print(f'gate: all 54 recorded delta_stats rows reproduce, {len(REC_DELTA.columns)} columns, {n_checked} values')

# gate: the recorded M2 and M3 rows (white.csv), recomputed the same way
IDX = boot_index(N_ROIS)
for r in REC_WHITE.itertuples():
    a, b = r.contrast.split(' - ')
    if r.check == 'M2':
        m = share_diff_boot(roi_sum(PER_RUN, r.rank_key, a, 'n_white_fp_top10'), roi_sum(PER_RUN, r.rank_key, a, 'n_fp_top10'), roi_sum(PER_RUN, r.rank_key, b, 'n_white_fp_top10'), roi_sum(PER_RUN, r.rank_key, b, 'n_fp_top10'), IDX)
        assert np.allclose([m['share_a'], m['share_b'], m['ci_low'], m['ci_high']], [r.share_a, r.share_b, r.ci_low, r.ci_high], rtol=1e-12, atol=0), f'M2 {r.rank_key} {r.contrast}'
    else:
        m = count_delta(roi_sum(PER_RUN, r.rank_key, a, 'n_white_tp_top30') - roi_sum(PER_RUN, r.rank_key, b, 'n_white_tp_top30'), IDX)
        assert (m['total'], m['wins'], m['losses']) == (r.total, r.wins, r.losses) and np.allclose([m['exact_p'], m['ci_low'], m['ci_high']], [r.exact_p, r.ci_low, r.ci_high], rtol=1e-12, atol=0), f'M3 {r.rank_key} {r.contrast}'
print(f'gate: all {len(REC_WHITE)} recorded M2 / M3 rows reproduce')


gate: all 54 recorded delta_stats rows reproduce, 30 columns, 1620 values


gate: all 20 recorded M2 / M3 rows reproduce


## 1. Precision per arm, and every recorded contrast read through the premises

The recorded contrasts are unchanged; only the reading column is new. Then the white-adjacency checks the source
notebook ran on `A3 - A0`, `A2 - A0` and `A1 - A0`, repeated on the product contrast `A3 - A1` and at three white
cut-offs:
- **M2** is the share of top-10 false positives that are white-adjacent;
- the **counts** split the top-10 false positives into white-adjacent and the rest, each with an exact sign-flip over
  ROIs;
- **M3** is the number of white-adjacent mitoses in the top 30.

"White-adjacent" means more than the cut-off share of the 61 px window has gray > 215 on the unclipped RGB.

In [5]:
# pooled precision per arm and axis
grp = PER_RUN.groupby(['rank_key', 'arm'], sort=False)
POOLED = pd.DataFrame({f'P@{k}': grp[f'tp_at_{k}'].sum() / (k * grp.size()) for k in BUDGETS}).unstack(0).reindex(list(ARM_NAMES))
print('pooled precision per arm (294 clicks):')
print(POOLED.to_string(float_format=lambda v: f'{v:.4f}'))

# the recorded contrasts, read through the product premises: the clip is the candidate production change, the band a benchmark-harness correction
READING = {
    'A3 - A1': 'clip, harness corrected on both sides: the product estimate',
    'A2 - A0': 'clip, harness uncorrected on both sides',
    'A3 - A0': 'clip + harness correction: the pre-registered primary',
    'A1 - A0': 'harness correction alone (band out)',
    'A3 - A2': 'harness correction, given the clip',
    'A2m - A2': 'floor level: ROI median vs multi-Otsu',
    'A1 - A1v': 'band implementation: floored vs the spec valid-only',
    'A3 - A3v': 'band implementation, given the clip',
    'A2m - A0': 'median-floor clip, harness uncorrected',
}
CONTRAST_TAB = DELTA.assign(reading=DELTA['pair'].map(READING))[['reading', 'holm_family', 'rank_key', 'pair', 'K', 'pooled_delta_precision', 'ci_low', 'ci_high', 't_ci_low', 't_ci_high', 'wins', 'losses', 'ties', 'exact_p', 'holm_p', 'd5_bar_met', 'significant_gain', 'significant_loss']]
order = list(READING)
CONTRAST_TAB = CONTRAST_TAB.assign(_o=CONTRAST_TAB['pair'].map(order.index)).sort_values(['_o', 'rank_key', 'K'], ascending=[True, False, True], kind='stable').drop(columns='_o').reset_index(drop=True)
pts = CONTRAST_TAB.copy()
for c in ('pooled_delta_precision', 'ci_low', 'ci_high', 't_ci_low', 't_ci_high'):
    pts[c] = 100 * pts[c]
print('\nevery recorded contrast, in precision points (x100), with its reading:')
print(pts.to_string(index=False, float_format=lambda v: f'{v:.3g}'))

# M2 and M3 on the product contrast A3 - A1, which the source notebook did not test; and M2 at other white cut-offs from the saved top 30
top10 = TOP[TOP['rank'] < 10]
assert top10.groupby(['file_name', 'seed_index', 'arm', 'rank_key']).size().eq(10).all()
check = top10.assign(fp=top10['bucket'] != TP, wfp=(top10['bucket'] != TP) & (top10['white_frac_61'] > WHITE_SHARE)).groupby(['rank_key', 'arm', 'file_name', 'seed_index'])[['fp', 'wfp']].sum()
rec = PER_RUN.set_index(['rank_key', 'arm', 'file_name', 'seed_index'])[['n_fp_top10', 'n_white_fp_top10']]
assert (check['fp'] == rec.loc[check.index, 'n_fp_top10']).all() and (check['wfp'] == rec.loc[check.index, 'n_white_fp_top10']).all(), 'top30.csv does not reproduce the per-run top-10 FP counts'


def top_counts(axis, arm, cut, k):
    """
        Per-ROI counts in the top K of one arm: false positives, false positives with white share above cut, and true positives with white share above cut.

        axis (str): the rank_key.
        arm (str): the arm.
        cut (float): the white-share cut-off.
        k (int): the list depth.

        Returns tuple[np.ndarray, np.ndarray, np.ndarray]: int64 per ROI in ROI_ORDER.
    """
    t = TOP[(TOP['rank_key'] == axis) & (TOP['arm'] == arm) & (TOP['rank'] < k)]
    fp, w = (t['bucket'] != TP), (t['white_frac_61'] > cut)
    g = t.assign(fp=fp, wfp=fp & w, wtp=~fp & w).groupby('file_name')[['fp', 'wfp', 'wtp']].sum().reindex(ROI_ORDER)
    return g['fp'].to_numpy(np.int64), g['wfp'].to_numpy(np.int64), g['wtp'].to_numpy(np.int64)


wc_rows = []
for axis in ('tm_score', 'chromatin_od'):
    for b, a in CLIP_PAIRS + (('A0', 'A3'), ('A0', 'A1')):
        for cut in WHITE_CUTS:
            fa, wa, _ = top_counts(axis, a, cut, 10)
            fb, wb, _ = top_counts(axis, b, cut, 10)
            m2 = share_diff_boot(wa, fa, wb, fb, IDX)
            wfp, nwfp = count_delta(wa - wb, IDX), count_delta((fa - wa) - (fb - wb), IDX)
            _, _, ta = top_counts(axis, a, cut, 30)
            _, _, tb = top_counts(axis, b, cut, 30)
            m3 = count_delta(ta - tb, IDX)
            wc_rows.append(dict(rank_key=axis, contrast=f'{a} - {b}', white_cut=cut, m2_share_b=m2['share_b'], m2_share_a=m2['share_a'], m2_diff=m2['diff'], m2_ci_low=m2['ci_low'], m2_ci_high=m2['ci_high'], m2_falls=m2['ci_below_0'], white_fp10_b=int(wb.sum()), white_fp10_a=int(wa.sum()), white_fp10_delta=wfp['total'], white_fp10_wlt=f"{wfp['wins']}/{wfp['losses']}/{wfp['ties']}", white_fp10_p=wfp['exact_p'], white_fp10_ci=f"[{wfp['ci_low']:.0f}, {wfp['ci_high']:.0f}]", nonwhite_fp10_b=int((fb - wb).sum()), nonwhite_fp10_a=int((fa - wa).sum()), nonwhite_fp10_delta=nwfp['total'], nonwhite_fp10_wlt=f"{nwfp['wins']}/{nwfp['losses']}/{nwfp['ties']}", nonwhite_fp10_p=nwfp['exact_p'], nonwhite_fp10_ci=f"[{nwfp['ci_low']:.0f}, {nwfp['ci_high']:.0f}]", m3_white_tp30_b=int(tb.sum()), m3_white_tp30_a=int(ta.sum()), m3_delta=m3['total'], m3_p=m3['exact_p'], m3_ci_low=m3['ci_low'], m3_ci_high=m3['ci_high']))
WHITE_CHECKS = pd.DataFrame(wc_rows)
# the pre-registered cut reproduces the recorded M2 rows
for r in REC_WHITE[(REC_WHITE['check'] == 'M2')].itertuples():
    w = WHITE_CHECKS[(WHITE_CHECKS['rank_key'] == r.rank_key) & (WHITE_CHECKS['contrast'] == r.contrast) & (WHITE_CHECKS['white_cut'] == WHITE_SHARE)]
    if len(w):
        assert np.allclose(w[['m2_share_a', 'm2_share_b', 'm2_ci_low', 'm2_ci_high']].to_numpy(float)[0], [r.share_a, r.share_b, r.ci_low, r.ci_high], rtol=1e-12, atol=0)
print('\nwhite-adjacent checks (top-10 FP share = M2; top-10 FP counts split white / non-white; top-30 white TPs = M3), per contrast and white cut-off:')
print(WHITE_CHECKS.to_string(index=False, float_format=lambda v: f'{v:.4g}'))


pooled precision per arm (294 clicks):
                 P@10                  P@20                  P@30         
rank_key chromatin_od tm_score chromatin_od tm_score chromatin_od tm_score
arm                                                                       
A0             0.5626   0.5418       0.5090   0.4685       0.4430   0.4162
A1             0.5854   0.5459       0.5179   0.4723       0.4499   0.4186
A1v            0.5850   0.5456       0.5177   0.4714       0.4493   0.4183
A2             0.5592   0.5738       0.5095   0.4969       0.4438   0.4395
A3             0.5806   0.5789       0.5187   0.4985       0.4508   0.4416
A3v            0.5810   0.5782       0.5189   0.4978       0.4505   0.4408
A2m            0.5578   0.5741       0.5066   0.4946       0.4435   0.4395

every recorded contrast, in precision points (x100), with its reading:
                                                    reading      holm_family     rank_key     pair  K  pooled_delta_precision  ci_low  ci_h


white-adjacent checks (top-10 FP share = M2; top-10 FP counts split white / non-white; top-30 white TPs = M3), per contrast and white cut-off:
    rank_key contrast  white_cut  m2_share_b  m2_share_a    m2_diff  m2_ci_low  m2_ci_high  m2_falls  white_fp10_b  white_fp10_a  white_fp10_delta white_fp10_wlt  white_fp10_p white_fp10_ci  nonwhite_fp10_b  nonwhite_fp10_a  nonwhite_fp10_delta nonwhite_fp10_wlt  nonwhite_fp10_p nonwhite_fp10_ci  m3_white_tp30_b  m3_white_tp30_a  m3_delta   m3_p  m3_ci_low  m3_ci_high
    tm_score  A3 - A1        0.1      0.4345      0.3837   -0.05077   -0.07105    -0.03038      True           580           475              -105        4/27/18     5.104e-07   [-148, -67]              755              763                    8          18/16/15           0.6648        [-23, 39]              832              818       -14  0.251        -37           6
    tm_score  A3 - A1        0.2      0.2367      0.1939   -0.04284   -0.06022    -0.02705      True           316

## 2. Pairs that should tie: how large a difference the data still allow

"Holm p = 1" is non-significance, not equivalence. For each pair expected to tie, the table gives the largest
difference that either 95 % interval (bootstrap or t) still allows, and how many of the 294 clicks have an identical
top K. No equivalence margin was set in advance, so read these bounds as descriptive.

In [6]:
# the pairs that should not differ: how large a difference the data still allow, in points (bootstrap and t intervals)
BOUND_PAIRS = [('tm_score', 'A2m - A2'), ('chromatin_od', 'A2m - A2'), ('tm_score', 'A1 - A1v'), ('chromatin_od', 'A1 - A1v'), ('tm_score', 'A3 - A3v'), ('chromatin_od', 'A3 - A3v'), ('tm_score', 'A1 - A0'), ('tm_score', 'A3 - A2')]
b_rows = []
for axis, pair in BOUND_PAIRS:
    a, b = CONTRASTS[pair]
    for k in BUDGETS:
        r = DELTA[(DELTA['rank_key'] == axis) & (DELTA['pair'] == pair) & (DELTA['K'] == k)].iloc[0]
        la = TOP[(TOP['rank_key'] == axis) & (TOP['arm'] == a) & (TOP['rank'] < k)].sort_values(['file_name', 'seed_index', 'rank'])
        lb = TOP[(TOP['rank_key'] == axis) & (TOP['arm'] == b) & (TOP['rank'] < k)].sort_values(['file_name', 'seed_index', 'rank'])
        same = (la[['cx', 'cy']].to_numpy() == lb[['cx', 'cy']].to_numpy()).all(axis=1).reshape(-1, k).all(axis=1)
        b_rows.append(dict(rank_key=axis, pair=pair, reading=READING[pair], K=k, delta_pts=100 * r['pooled_delta_precision'], boot_ci_pts=f"[{100 * r['ci_low']:+.2f}, {100 * r['ci_high']:+.2f}]", t_ci_pts=f"[{100 * r['t_ci_low']:+.2f}, {100 * r['t_ci_high']:+.2f}]", largest_diff_not_excluded_pts=100 * max(abs(r['ci_low']), abs(r['ci_high']), abs(r['t_ci_low']), abs(r['t_ci_high'])), wlt=f"{r['wins']}/{r['losses']}/{r['ties']}", holm_p=r['holm_p'], clicks_identical_topK=int(same.sum()), clicks=len(same)))
BOUNDS = pd.DataFrame(b_rows)
print('pairs expected to tie: the delta, both intervals, the largest difference either interval still allows, and how many of 294 clicks have an identical top K:')
print(BOUNDS.to_string(index=False, float_format=lambda v: f'{v:.3g}'))

# the floors themselves: b_med clips 50 % by construction; b_otsu clips a ROI-dependent share
fl = FLOORS[['file_name', 'domain', 'b_otsu', 'b_med', 'clip_share_otsu', 'clip_share_med']].assign(otsu_over_med=FLOORS['b_otsu'] / FLOORS['b_med'])
print('\nfloors over the 49 ROIs (b_otsu / b_med and the share each clips):')
print(fl[['b_otsu', 'b_med', 'otsu_over_med', 'clip_share_otsu', 'clip_share_med']].describe().loc[['mean', 'min', '50%', 'max']].to_string(float_format=lambda v: f'{v:.3f}'))
print(f"b_otsu > b_med on {int((fl['b_otsu'] > fl['b_med']).sum())}/49 ROIs")


pairs expected to tie: the delta, both intervals, the largest difference either interval still allows, and how many of 294 clicks have an identical top K:
    rank_key     pair                                             reading  K  delta_pts    boot_ci_pts       t_ci_pts  largest_diff_not_excluded_pts      wlt  holm_p  clicks_identical_topK  clicks
    tm_score A2m - A2               floor level: ROI median vs multi-Otsu 10      0.034 [-0.44, +0.51] [-0.47, +0.54]                          0.537 13/11/25       1                     33     294
    tm_score A2m - A2               floor level: ROI median vs multi-Otsu 20     -0.238 [-0.56, +0.10] [-0.58, +0.11]                          0.584 10/21/18       1                      8     294
    tm_score A2m - A2               floor level: ROI median vs multi-Otsu 30          0 [-0.24, +0.23] [-0.23, +0.23]                          0.238 15/15/19       1                      3     294
chromatin_od A2m - A2               floor level: ROI medi

## 3. Top-10 churn by object, not by pixel

The clip changes the image and the template, so almost every surviving peak moves by a pixel. A top-10 candidate
counts as the same object when the other arm's top 10 has one within `tol` px (Chebyshev). `tol = 0` is the source
notebook's count. Mitoses are also counted by annotation id.

The audit counted `A0 -> A3` and `A0 -> A2`, and its table is the cross-check here. The product contrast is
`A1 -> A3`; `A0 -> A1` (the band alone) is shown for scale.

In [7]:
def near_any(xy, other, tol):
    """
        Which points have a point of another set within tol px (Chebyshev).

        xy (np.ndarray): (n, 2) points.
        other (np.ndarray): (m, 2) points.
        tol (int): the tolerance in px.

        Returns np.ndarray: bool, one per point of xy.
    """
    if len(xy) == 0 or len(other) == 0:
        return np.zeros(len(xy), dtype=bool)
    return (np.max(np.abs(xy[:, None, :] - other[None, :, :]), axis=2) <= tol).any(axis=1)


def top10_index(axis):
    """
        Every list's top 10 on one axis, keyed by click and arm.

        axis (str): the rank_key.

        Returns dict: (file_name, seed_index, arm) -> the top-10 rows in rank order.
    """
    t = TOP[(TOP['rank_key'] == axis) & (TOP['rank'] < 10)]
    return {key: g.sort_values('rank') for key, g in t.groupby(['file_name', 'seed_index', 'arm'])}


def churn_click(ta, tb, tol):
    """
        Top-10 composition change from one arm's list to another's on one click, a candidate kept when the other list has one within tol px (Chebyshev).

        ta (pd.DataFrame): the base arm's top 10.
        tb (pd.DataFrame): the new arm's top 10.
        tol (int): coordinate tolerance in px; 0 is exact equality.

        Returns dict: slots changed; mitoses, false positives and white-adjacent false positives leaving and entering; mitoses also by annotation id.
    """
    xa, xb = ta[['cx', 'cy']].to_numpy(), tb[['cx', 'cy']].to_numpy()
    near = np.max(np.abs(xa[:, None, :] - xb[None, :, :]), axis=2) <= tol
    leave, enter = ~near.any(axis=1), ~near.any(axis=0)
    tpa, tpb = (ta['bucket'] == TP).to_numpy(), (tb['bucket'] == TP).to_numpy()
    wa, wb = (ta['white_frac_61'] > WHITE_SHARE).to_numpy(), (tb['white_frac_61'] > WHITE_SHARE).to_numpy()
    ida, idb = set(ta.loc[tpa, 'matched_ann_id']), set(tb.loc[tpb, 'matched_ann_id'])
    return dict(slots_changed=int(leave.sum()), mit_leave=int((leave & tpa).sum()), mit_enter=int((enter & tpb).sum()), fp_leave=int((leave & ~tpa).sum()), fp_enter=int((enter & ~tpb).sum()), white_fp_leave=int((leave & ~tpa & wa).sum()), white_fp_enter=int((enter & ~tpb & wb).sum()), nonwhite_fp_leave=int((leave & ~tpa & ~wa).sum()), nonwhite_fp_enter=int((enter & ~tpb & ~wb).sum()), mit_leave_by_id=len(ida - idb), mit_enter_by_id=len(idb - ida), fp_base=int((~tpa).sum()), white_fp_base=int((~tpa & wa).sum()))


T10 = top10_index(PRODUCT_AXIS)
CLICK_KEYS = sorted({(f, s) for f, s, _ in T10})
assert len(CLICK_KEYS) == 294
churn_rows = []
for b, a in (('A0', 'A3'), ('A0', 'A2'), ('A1', 'A3'), ('A0', 'A1')):
    for tol in TOLS:
        for f, s in CLICK_KEYS:
            churn_rows.append(dict(arm_from=b, arm_to=a, tol_px=tol, file_name=f, seed_index=s, domain=DOMAIN_OF[f], **churn_click(T10[(f, s, b)], T10[(f, s, a)], tol)))
CHURN_CLICK = pd.DataFrame(churn_rows)
CHURN = CHURN_CLICK.groupby(['arm_from', 'arm_to', 'tol_px'], sort=False).sum(numeric_only=True).drop(columns='seed_index').reset_index()
CHURN['white_share_base_fp'] = CHURN['white_fp_base'] / CHURN['fp_base']
CHURN['white_share_leaving_fp'] = CHURN['white_fp_leave'] / CHURN['fp_leave']
CHURN['white_share_entering_fp'] = CHURN['white_fp_enter'] / CHURN['fp_enter']

# cross-check: this independent count equals the audit's on the two pairs it counted, at every shared tolerance
aud = pd.read_csv(INPUTS['audit_churn_top10'])
for r in aud[aud['tol_px'].isin(TOLS)].itertuples():
    m = CHURN[(CHURN['arm_from'] == r.arm_from) & (CHURN['arm_to'] == r.arm_to) & (CHURN['tol_px'] == r.tol_px)].iloc[0]
    assert [m[c] for c in ('slots_changed', 'mit_leave', 'mit_enter', 'fp_leave', 'fp_enter', 'white_fp_leave', 'white_fp_enter', 'mit_leave_by_id', 'mit_enter_by_id')] == [r.slots_changed, r.mit_leave, r.mit_enter, r.fp_leave, r.fp_enter, r.white_fp_leave, r.white_fp_enter, r.mit_leave_by_ann_id, r.mit_enter_by_ann_id], f'churn differs from the audit on {r.arm_from}->{r.arm_to} tol {r.tol_px}'
# and the net counts are the per-run deltas, whatever the tolerance
for (b, a, tol), g in CHURN_CLICK.groupby(['arm_from', 'arm_to', 'tol_px']):
    d_tp = int(roi_sum(PER_RUN, PRODUCT_AXIS, a, 'tp_at_10').sum() - roi_sum(PER_RUN, PRODUCT_AXIS, b, 'tp_at_10').sum())
    d_wfp = int(roi_sum(PER_RUN, PRODUCT_AXIS, a, 'n_white_fp_top10').sum() - roi_sum(PER_RUN, PRODUCT_AXIS, b, 'n_white_fp_top10').sum())
    assert g['mit_enter'].sum() - g['mit_leave'].sum() == d_tp == g['mit_enter_by_id'].sum() - g['mit_leave_by_id'].sum() and g['white_fp_enter'].sum() - g['white_fp_leave'].sum() == d_wfp
print(f'churn cross-check: equal to the audit table on A0->A3 and A0->A2 at tolerances {TOLS}; every net equals the per-run delta')
show = ['arm_from', 'arm_to', 'tol_px', 'slots_changed', 'mit_leave', 'mit_enter', 'mit_leave_by_id', 'mit_enter_by_id', 'fp_leave', 'fp_enter', 'white_fp_leave', 'white_fp_enter', 'nonwhite_fp_leave', 'nonwhite_fp_enter', 'white_share_base_fp', 'white_share_leaving_fp', 'white_share_entering_fp']
print('\ntop-10 churn at tm_score, 294 clicks (2940 slots); tol 0 is the source notebook\'s exact-coordinate count:')
print(CHURN[show].to_string(index=False, float_format=lambda v: f'{v:.3f}'))

# per domain, the product contrast at 1 px: which top-10 false positives the clip removes
dom = CHURN_CLICK[(CHURN_CLICK['tol_px'] == 1)].groupby(['arm_from', 'arm_to', 'domain'], sort=False)[['fp_base', 'white_fp_base', 'fp_leave', 'fp_enter', 'white_fp_leave', 'white_fp_enter', 'nonwhite_fp_leave', 'nonwhite_fp_enter']].sum().reset_index()
dom['net_white_fp'] = dom['white_fp_enter'] - dom['white_fp_leave']
dom['net_nonwhite_fp'] = dom['nonwhite_fp_enter'] - dom['nonwhite_fp_leave']
dom['white_fp_removed_frac'] = -dom['net_white_fp'] / dom['white_fp_base']
dom['nonwhite_fp_removed_frac'] = -dom['net_nonwhite_fp'] / (dom['fp_base'] - dom['white_fp_base'])
dom['domain'] = dom['domain'].map(SHORT)
CHURN_DOMAIN = dom
print('\nper domain, 1 px identity: the net change in white-adjacent and other top-10 false positives, and the share of each the clip removes:')
print(dom[dom['arm_from'].isin(['A1', 'A0']) & dom['arm_to'].isin(['A3', 'A2']) & ~((dom['arm_from'] == 'A0') & (dom['arm_to'] == 'A3'))].to_string(index=False, float_format=lambda v: f'{v:.3f}'))


churn cross-check: equal to the audit table on A0->A3 and A0->A2 at tolerances (0, 1, 3); every net equals the per-run delta

top-10 churn at tm_score, 294 clicks (2940 slots); tol 0 is the source notebook's exact-coordinate count:
arm_from arm_to  tol_px  slots_changed  mit_leave  mit_enter  mit_leave_by_id  mit_enter_by_id  fp_leave  fp_enter  white_fp_leave  white_fp_enter  nonwhite_fp_leave  nonwhite_fp_enter  white_share_base_fp  white_share_leaving_fp  white_share_entering_fp
      A0     A3       0           1193        560        669              128              237       633       524             159              77                474                447                0.239                   0.251                    0.147
      A0     A3       1            484        159        268              128              237       325       216             103              21                222                195                0.239                   0.317                    0.097
   

## 4. Dose-response: which top-10 candidates the clip moves out, by white share

Take the base arm's top-10 candidates and bin them by the white share of their 61 px window. The table gives the share
of each bin that leaves the top 10 under the clip (1 px identity, with a ROI-cluster bootstrap CI). It also gives how
many candidates of each bin enter, and the net.

The mechanism predicts that the leave rate of false positives rises with white share. It also predicts that, if the
clip only acted on the surround, white-adjacent mitoses would leave at the same raised rate.

In [8]:
# dose-response: of the base arm's top-10 candidates in each white-share bin, how many the clip moves out of the top 10 (1 px identity), with a ROI-cluster bootstrap CI
WHITE_BINS = (0.0, 0.02, 0.10, 0.20, 0.40, 1.0)
BIN_LABELS = ['<=2%', '2-10%', '10-20%', '20-40%', '>40%']


def flagged_top10(b, a, tol):
    """
        The base arm's top-10 rows, each flagged by whether the new arm's top 10 has a candidate within tol px; and the new arm's rows, flagged the same way.

        b (str): the base arm.
        a (str): the new arm.
        tol (int): coordinate tolerance in px (Chebyshev).

        Returns tuple[pd.DataFrame, pd.DataFrame]: base rows with 'leaves', new rows with 'enters'.
    """
    base, new = [], []
    for f, s in CLICK_KEYS:
        tb, ta = T10[(f, s, b)], T10[(f, s, a)]
        base.append(tb.assign(leaves=~near_any(tb[['cx', 'cy']].to_numpy(), ta[['cx', 'cy']].to_numpy(), tol)))
        new.append(ta.assign(enters=~near_any(ta[['cx', 'cy']].to_numpy(), tb[['cx', 'cy']].to_numpy(), tol)))
    return pd.concat(base, ignore_index=True), pd.concat(new, ignore_index=True)


def rate_by_bin(frame, flag, idx):
    """
        The share of rows carrying a flag in each white-share bin, pooled over clicks, with a ROI-cluster bootstrap 95% CI.

        frame (pd.DataFrame): rows with file_name, white_frac_61 and the flag column.
        flag (str): the boolean column to rate.
        idx (np.ndarray): (N_BOOT, N_ROIS) resampled ROI indices.

        Returns pd.DataFrame: one row per bin with n, the flagged count, the rate and its CI.
    """
    fb = frame.assign(wbin=pd.cut(frame['white_frac_61'], WHITE_BINS, labels=BIN_LABELS, include_lowest=True))
    rows = []
    for lab in BIN_LABELS:
        g = fb[fb['wbin'] == lab].groupby('file_name')[flag].agg(['size', 'sum']).reindex(ROI_ORDER, fill_value=0)
        n, k = g['size'].to_numpy(np.int64), g['sum'].to_numpy(np.int64)
        nb, kb = n[idx].sum(axis=1), k[idx].sum(axis=1)
        ok = nb > 0
        lo, hi = np.percentile(kb[ok] / nb[ok], [2.5, 97.5]) if ok.any() else (np.nan, np.nan)
        rows.append(dict(white_bin=lab, n=int(n.sum()), flagged=int(k.sum()), rate=k.sum() / n.sum() if n.sum() else np.nan, ci_low=float(lo), ci_high=float(hi), rois_with_any=int((n > 0).sum())))
    return pd.DataFrame(rows)


DOSE = []
for b, a in CLIP_PAIRS + (('A0', 'A1'),):
    base, new = flagged_top10(b, a, 1)
    for cls, sel_b, sel_n in (('false positive', base['bucket'] != TP, new['bucket'] != TP), ('mitosis', base['bucket'] == TP, new['bucket'] == TP)):
        lv = rate_by_bin(base[sel_b], 'leaves', IDX).add_prefix('leave_').rename(columns={'leave_white_bin': 'white_bin'})
        en = new[sel_n].assign(wbin=pd.cut(new.loc[sel_n, 'white_frac_61'], WHITE_BINS, labels=BIN_LABELS, include_lowest=True)).groupby('wbin', observed=False)['enters'].sum().reindex(BIN_LABELS).to_numpy()
        DOSE.append(lv.assign(contrast=f'{a} - {b}', cls=cls, enter_n=en, net=en - lv['leave_flagged']))
DOSE = pd.concat(DOSE, ignore_index=True)[['contrast', 'cls', 'white_bin', 'leave_n', 'leave_flagged', 'leave_rate', 'leave_ci_low', 'leave_ci_high', 'leave_rois_with_any', 'enter_n', 'net']]
for (c, cls), g in DOSE.groupby(['contrast', 'cls'], sort=False):
    assert g['net'].sum() == CHURN.set_index(['arm_from', 'arm_to', 'tol_px']).loc[(c.split(' - ')[1], c.split(' - ')[0], 1), 'mit_enter' if cls == 'mitosis' else 'fp_enter'] - CHURN.set_index(['arm_from', 'arm_to', 'tol_px']).loc[(c.split(' - ')[1], c.split(' - ')[0], 1), 'mit_leave' if cls == 'mitosis' else 'fp_leave']
print('top-10 candidates at tm_score by white share of their 61 px window: how many leave the top 10 under the contrast (1 px identity, ROI-cluster CI), how many enter, and the net:')
print(DOSE.to_string(index=False, float_format=lambda v: f'{v:.3f}'))


# within a white bin, do false positives leave faster than mitoses? Candidates near rank 10 leave far more often than those near rank 1, and false
# positives sit slightly lower, so the gap is also given rank-standardised: the mitosis leave rate per rank stratum, weighted by the false positives' strata
RANK_STRATA = ((0, 4), (5, 7), (8, 9))
base, _ = flagged_top10('A1', 'A3', 1)
base = base.assign(wbin=pd.cut(base['white_frac_61'], WHITE_BINS, labels=BIN_LABELS, include_lowest=True), fp=base['bucket'] != TP, stratum=np.select([base['rank'].between(lo, hi) for lo, hi in RANK_STRATA], range(len(RANK_STRATA))))
print(f"mean base rank (0 = top): false positives {base.loc[base['fp'], 'rank'].mean():.2f}, mitoses {base.loc[~base['fp'], 'rank'].mean():.2f}")
gap_rows = []
for lab in BIN_LABELS:
    g = base[base['wbin'] == lab]
    cnt = {}
    for cls, is_fp in (('f', True), ('m', False)):
        t = g[g['fp'] == is_fp].groupby(['file_name', 'stratum'])['leaves'].agg(['size', 'sum']).unstack('stratum', fill_value=0).reindex(index=ROI_ORDER, fill_value=0)
        cnt[f'n_{cls}'] = t['size'].reindex(columns=range(len(RANK_STRATA)), fill_value=0).to_numpy(np.int64)
        cnt[f'k_{cls}'] = t['sum'].reindex(columns=range(len(RANK_STRATA)), fill_value=0).to_numpy(np.int64)


    def gaps(n_f, k_f, n_m, k_m):
        """
            The raw and the rank-standardised false-positive minus mitosis leave rate, from per-stratum counts.

            n_f (np.ndarray): false positives per rank stratum, (..., S).
            k_f (np.ndarray): false positives that leave, per stratum.
            n_m (np.ndarray): mitoses per stratum.
            k_m (np.ndarray): mitoses that leave, per stratum.

            Returns tuple[np.ndarray, np.ndarray, np.ndarray]: the raw gap, the standardised gap, and whether every stratum holding a false positive holds a mitosis.
        """
        fp_rate = k_f.sum(-1) / n_f.sum(-1)
        with np.errstate(divide='ignore', invalid='ignore'):
            m_s = np.where(n_m > 0, k_m / np.maximum(n_m, 1), 0.0)
        std_mit = (n_f * m_s).sum(-1) / n_f.sum(-1)
        ok = ((n_f == 0) | (n_m > 0)).all(-1) & (n_f.sum(-1) > 0) & (n_m.sum(-1) > 0)
        return fp_rate - k_m.sum(-1) / np.maximum(n_m.sum(-1), 1), fp_rate - std_mit, ok


    raw, std, ok = gaps(*(cnt[c].sum(axis=0) for c in ('n_f', 'k_f', 'n_m', 'k_m')))
    b_raw, b_std, b_ok = gaps(*(cnt[c][IDX].sum(axis=1) for c in ('n_f', 'k_f', 'n_m', 'k_m')))
    lo_r, hi_r = np.percentile(b_raw[b_ok], [2.5, 97.5])
    lo_s, hi_s = np.percentile(b_std[b_ok], [2.5, 97.5])
    gap_rows.append(dict(white_bin=lab, fp_leave_rate=cnt['k_f'].sum() / cnt['n_f'].sum(), mit_leave_rate=cnt['k_m'].sum() / cnt['n_m'].sum(), gap_raw=float(raw), gap_raw_ci_low=float(lo_r), gap_raw_ci_high=float(hi_r), gap_rank_std=float(std), gap_rank_std_ci_low=float(lo_s), gap_rank_std_ci_high=float(hi_s), all_strata_ok=bool(ok), n_boot_used=int(b_ok.sum()), **{f'fp_rate_rank{lo}-{hi}': cnt['k_f'][:, i].sum() / max(cnt['n_f'][:, i].sum(), 1) for i, (lo, hi) in enumerate(RANK_STRATA)}, **{f'mit_rate_rank{lo}-{hi}': cnt['k_m'][:, i].sum() / max(cnt['n_m'][:, i].sum(), 1) for i, (lo, hi) in enumerate(RANK_STRATA)}))
DOSE_GAP = pd.DataFrame(gap_rows)
print('\nA3 - A1: false-positive minus mitosis leave rate within each white bin, raw and rank-standardised (strata 0-4 / 5-7 / 8-9), ROI-cluster CIs; and the leave rates per stratum:')
print(DOSE_GAP.to_string(index=False, float_format=lambda v: f'{v:.3f}'))


top-10 candidates at tm_score by white share of their 61 px window: how many leave the top 10 under the contrast (1 px identity, ROI-cluster CI), how many enter, and the net:
contrast            cls white_bin  leave_n  leave_flagged  leave_rate  leave_ci_low  leave_ci_high  leave_rois_with_any  enter_n  net
 A3 - A1 false positive      <=2%      359             49       0.136         0.094          0.175                   30       80   31
 A3 - A1 false positive     2-10%      396             90       0.227         0.178          0.276                   43       66  -24
 A3 - A1 false positive    10-20%      264             55       0.208         0.159          0.264                   33       27  -28
 A3 - A1 false positive    20-40%      237             72       0.304         0.228          0.408                   29       17  -55
 A3 - A1 false positive      >40%       79             22       0.278         0.205          0.395                   12        1  -21
 A3 - A1        mitos

mean base rank (0 = top): false positives 4.80, mitoses 4.25



A3 - A1: false-positive minus mitosis leave rate within each white bin, raw and rank-standardised (strata 0-4 / 5-7 / 8-9), ROI-cluster CIs; and the leave rates per stratum:
white_bin  fp_leave_rate  mit_leave_rate  gap_raw  gap_raw_ci_low  gap_raw_ci_high  gap_rank_std  gap_rank_std_ci_low  gap_rank_std_ci_high  all_strata_ok  n_boot_used  fp_rate_rank0-4  fp_rate_rank5-7  fp_rate_rank8-9  mit_rate_rank0-4  mit_rate_rank5-7  mit_rate_rank8-9
     <=2%          0.136           0.086    0.051           0.004            0.093         0.036               -0.017                 0.083           True        10000            0.054            0.128            0.310             0.022             0.105             0.250
    2-10%          0.227           0.103    0.125           0.063            0.184         0.111                0.055                 0.167           True        10000            0.040            0.274            0.535             0.004             0.158             0.280
   10-

## 5. Pool and chromatin-od flows by object

Each pool is re-matched with `ev.bucket_detections` (D6), and the buckets must equal the recorded ones.
- **Pool mitoses** are counted by annotation id, matched in `tm_score` order. That is what `n_mit_pool` counts.
- **"New to the pool"** for a mitosis means no detection of the base pool lies within the match radius of it. This
  coverage test does not depend on matching order.
- **Non-mitoses** are counted by position (1 and 3 px).

In [9]:
images, annotations = ds.load_annotations(DB_PATH)
seed_of = PER_RUN.drop_duplicates(['file_name', 'seed_index']).set_index(['file_name', 'seed_index'])[['seed_ann_id', 'match_radius_px']]
pool_ids, pool_cov, pool_xy, pool_tp = {}, {}, {}, {}
gt_cache = {}
for (f, s, arm), g in POOL[POOL['arm'].isin(FLOW_ARMS)].groupby(['file_name', 'seed_index', 'arm']):
    g = g.sort_values('rank').reset_index(drop=True)
    if f not in gt_cache:
        gt_cache[f] = ds.image_annotations(annotations, f)
    gt = gt_cache[f]
    gt_eval = gt[gt['ann_id'] != seed_of.loc[(f, s), 'seed_ann_id']].reset_index(drop=True)
    radius = float(seed_of.loc[(f, s), 'match_radius_px'])
    out, _ = ev.bucket_detections(g[['cx', 'cy']], gt_eval, radius)
    assert (out['bucket'].to_numpy() == g['bucket'].to_numpy()).all(), f'{f} s{s} {arm}: the pool does not re-match to its recorded buckets'
    is_tp = (out['bucket'] == TP).to_numpy()
    mit = gt_eval[gt_eval['category_id'] == ds.MITOTIC]
    dist = np.hypot(mit['cx'].to_numpy()[:, None] - g['cx'].to_numpy()[None, :], mit['cy'].to_numpy()[:, None] - g['cy'].to_numpy()[None, :])
    pool_ids[(f, s, arm)] = set(out.loc[is_tp, 'matched_ann_id'])
    pool_cov[(f, s, arm)] = set(mit['ann_id'].to_numpy()[(dist <= radius).any(axis=1)])
    assert pool_ids[(f, s, arm)] <= pool_cov[(f, s, arm)]
    pool_xy[(f, s, arm)] = g[['cx', 'cy']].to_numpy()
    pool_tp[(f, s, arm)] = is_tp
assert len(pool_ids) == 294 * len(FLOW_ARMS)
n_mit = PER_RUN[PER_RUN['rank_key'] == 'tm_score'].set_index(['file_name', 'seed_index', 'arm'])['n_mit_pool']
assert all(len(v) == n_mit[k] for k, v in pool_ids.items()), 'the re-matched pool mitoses differ from n_mit_pool'
print(f'{len(pool_ids)} pools re-matched with ev.bucket_detections: buckets equal the recorded ones and mitosis counts equal n_mit_pool')


T10_OD = top10_index('chromatin_od')
flow_rows = []
for b, a in (('A0', 'A2'), ('A1', 'A3'), ('A0', 'A3')):
    c = dict(arm_from=b, arm_to=a, pool_mit_leave_by_id=0, pool_mit_enter_by_id=0, pool_mit_net=0, pool_nonmit_leave_1px=0, pool_nonmit_enter_1px=0, pool_nonmit_leave_3px=0, pool_nonmit_enter_3px=0, od10_mit_enter_by_id=0, od10_mit_enter_new_to_pool=0, od10_mit_leave_by_id=0, od10_mit_leave_gone_from_pool=0, od10_nonmit_enter_3px=0, od10_nonmit_enter_new_to_pool_3px=0, od10_nonmit_leave_3px=0, od10_nonmit_leave_gone_from_pool_3px=0, od10_tp_delta=0)
    for f, s in CLICK_KEYS:
        ib, ia = pool_ids[(f, s, b)], pool_ids[(f, s, a)]
        c['pool_mit_leave_by_id'] += len(ib - ia)
        c['pool_mit_enter_by_id'] += len(ia - ib)
        c['pool_mit_net'] += len(ia) - len(ib)
        xb, xa = pool_xy[(f, s, b)], pool_xy[(f, s, a)]
        nb, na = ~pool_tp[(f, s, b)], ~pool_tp[(f, s, a)]
        for tol in (1, 3):
            c[f'pool_nonmit_leave_{tol}px'] += int((nb & ~near_any(xb, xa, tol)).sum())
            c[f'pool_nonmit_enter_{tol}px'] += int((na & ~near_any(xa, xb, tol)).sum())
        ob, oa = T10_OD[(f, s, b)], T10_OD[(f, s, a)]
        tb_, ta_ = (ob['bucket'] == TP).to_numpy(), (oa['bucket'] == TP).to_numpy()
        idb, ida = set(ob.loc[tb_, 'matched_ann_id']), set(oa.loc[ta_, 'matched_ann_id'])
        c['od10_mit_enter_by_id'] += len(ida - idb)
        c['od10_mit_enter_new_to_pool'] += len((ida - idb) - pool_cov[(f, s, b)])
        c['od10_mit_leave_by_id'] += len(idb - ida)
        c['od10_mit_leave_gone_from_pool'] += len((idb - ida) - pool_cov[(f, s, a)])
        c['od10_tp_delta'] += int(ta_.sum() - tb_.sum())
        xob, xoa = ob[['cx', 'cy']].to_numpy(), oa[['cx', 'cy']].to_numpy()
        ent = ~ta_ & ~near_any(xoa, xob, 3)
        lv = ~tb_ & ~near_any(xob, xoa, 3)
        c['od10_nonmit_enter_3px'] += int(ent.sum())
        c['od10_nonmit_enter_new_to_pool_3px'] += int((ent & ~near_any(xoa, xb, 3)).sum())
        c['od10_nonmit_leave_3px'] += int(lv.sum())
        c['od10_nonmit_leave_gone_from_pool_3px'] += int((lv & ~near_any(xob, xa, 3)).sum())
    assert c['od10_mit_enter_by_id'] - c['od10_mit_leave_by_id'] == c['od10_tp_delta'] == int(roi_sum(PER_RUN, 'chromatin_od', a, 'tp_at_10').sum() - roi_sum(PER_RUN, 'chromatin_od', b, 'tp_at_10').sum())
    flow_rows.append(c)
FLOWS = pd.DataFrame(flow_rows)
aud_pool = pd.read_csv(INPUTS['audit_churn_pool'])
r = aud_pool[(aud_pool['arm_from'] == 'A0') & (aud_pool['arm_to'] == 'A2')].iloc[0]
f0 = FLOWS[(FLOWS['arm_from'] == 'A0') & (FLOWS['arm_to'] == 'A2')].iloc[0]
assert (f0['pool_mit_leave_by_id'], f0['pool_mit_enter_by_id']) == (r.mit_leave_by_ann_id, r.mit_enter_by_ann_id), 'pool flows differ from the audit'
print('pool flows equal the audit on A0->A2 (by annotation id); od top-10 nets equal the per-run deltas')
print('\npool and chromatin-od top-10 flows by object (mitoses by annotation id; non-mitoses by position):')
print(FLOWS.T.to_string())


1176 pools re-matched with ev.bucket_detections: buckets equal the recorded ones and mitosis counts equal n_mit_pool


pool flows equal the audit on A0->A2 (by annotation id); od top-10 nets equal the per-run deltas

pool and chromatin-od top-10 flows by object (mitoses by annotation id; non-mitoses by position):
                                         0     1     2
arm_from                                A0    A1    A0
arm_to                                  A2    A3    A3
pool_mit_leave_by_id                   168   163   205
pool_mit_enter_by_id                   604   611   657
pool_mit_net                           436   448   452
pool_nonmit_leave_1px                 4151  4213  4496
pool_nonmit_enter_1px                 3637  3685  3961
pool_nonmit_leave_3px                 3784  3848  4113
pool_nonmit_enter_3px                 3270  3320  3578
od10_mit_enter_by_id                    49    58   109
od10_mit_enter_new_to_pool              47    55    57
od10_mit_leave_by_id                    59    72    56
od10_mit_leave_gone_from_pool            2     4    14
od10_nonmit_enter_3px             

## 6. Per domain, and the domain as the unit

There are 7 ROIs in each of 7 domains. At the domain level, the exact sign-flip over 7 units has a floor of
1/64 = 0.0156, and Holm over the three K raises that floor to 0.047.

In [10]:
DOM_ROWS = []
for axis in ('tm_score', 'chromatin_od'):
    for d in DOMAINS:
        fs = [f for f in ROI_ORDER if DOMAIN_OF[f] == d]
        sel = PER_RUN[(PER_RUN['rank_key'] == axis) & PER_RUN['file_name'].isin(fs)]
        row = dict(rank_key=axis, domain=SHORT[d], clip_share_otsu=FLOORS.set_index('file_name').loc[fs, 'clip_share_otsu'].mean(), white_share_roi=FLOORS.set_index('file_name').loc[fs, 'white_share_roi'].mean())
        for arm in ('A0', 'A1', 'A2', 'A3'):
            row[f'{arm}_P@10'] = sel.loc[sel['arm'] == arm, 'tp_at_10'].sum() / (10 * 6 * len(fs))
        for b, a in (('A1', 'A3'), ('A0', 'A2'), ('A0', 'A3')):
            dd = sel[sel['arm'] == a].groupby('file_name')['tp_at_10'].sum() - sel[sel['arm'] == b].groupby('file_name')['tp_at_10'].sum()
            row[f'{a}-{b}_pts'] = 100 * dd.sum() / (10 * 6 * len(fs))
            row[f'{a}-{b}_wlt'] = f'{int((dd > 0).sum())}/{int((dd < 0).sum())}/{int((dd == 0).sum())}'
        DOM_ROWS.append(row)
DOMAIN_TAB = pd.DataFrame(DOM_ROWS)
print('per domain, P@10 per arm and the clip contrasts in points, W/L/T over the domain\'s 7 ROIs:')
print(DOMAIN_TAB.to_string(index=False, float_format=lambda v: f'{v:.3f}'))

# the domain as the unit: 7 click-summed integer deltas, exact sign-flip (floor 2/128 = 1/64), Holm over the three K
dl_rows = []
for axis in ('tm_score', 'chromatin_od'):
    for b, a in (('A1', 'A3'), ('A0', 'A2'), ('A0', 'A3'), ('A0', 'A1')):
        ps = []
        for k in BUDGETS:
            sel = PER_RUN[PER_RUN['rank_key'] == axis]
            dd = sel[sel['arm'] == a].groupby('domain')[f'tp_at_{k}'].sum() - sel[sel['arm'] == b].groupby('domain')[f'tp_at_{k}'].sum()
            p, nz = exact_sign_flip_p(dd.to_numpy())
            ps.append(p)
            dl_rows.append(dict(rank_key=axis, contrast=f'{a} - {b}', K=k, domains_pos=int((dd > 0).sum()), domains_neg=int((dd < 0).sum()), domains_zero=int((dd == 0).sum()), exact_p=p, min_attainable_p=2.0 ** (1 - nz) if nz else 1.0))
        for r_, h in zip(dl_rows[-3:], holm_adjust(ps)):
            r_['holm_p_over_K'] = h
DOMAIN_LEVEL = pd.DataFrame(dl_rows)
print('\nthe domain as the unit (G = 7):')
print(DOMAIN_LEVEL.to_string(index=False, float_format=lambda v: f'{v:.4g}'))


per domain, P@10 per arm and the clip contrasts in points, W/L/T over the domain's 7 ROIs:
    rank_key              domain  clip_share_otsu  white_share_roi  A0_P@10  A1_P@10  A2_P@10  A3_P@10  A3-A1_pts A3-A1_wlt  A2-A0_pts A2-A0_wlt  A3-A0_pts A3-A0_wlt
    tm_score           mast cell            0.641            0.215    0.648    0.652    0.664    0.669      1.667     5/1/1      1.667     5/1/1      2.143     5/1/1
    tm_score                lung            0.499            0.073    0.376    0.374    0.419    0.417      4.286     4/0/3      4.286     4/0/3      4.048     4/1/2
    tm_score       lymphosarcoma            0.392            0.039    0.605    0.605    0.664    0.674      6.905     5/0/2      5.952     5/1/1      6.905     6/1/0
    tm_score soft tissue sarcoma            0.550            0.040    0.576    0.579    0.607    0.610      3.095     5/1/1      3.095     5/1/1      3.333     5/1/1
    tm_score              breast            0.648            0.096    0.631    

## 7. Per click, and the worst click per ROI

Per-seed variance is a product defect even when the mean is good, so this section shows how the clip changes the
per-click distribution and the worst click.

In [11]:
pc_rows = []
w = PER_RUN[PER_RUN['rank_key'] == PRODUCT_AXIS].pivot_table(index=['file_name', 'seed_index'], columns='arm', values='tp_at_10')
for b, a in (('A1', 'A3'), ('A0', 'A2'), ('A0', 'A3'), ('A0', 'A1')):
    d = (w[a] - w[b])
    worst_b, worst_a = w[b].groupby('file_name').min(), w[a].groupby('file_name').min()
    pc_rows.append(dict(contrast=f'{a} - {b}', clicks_better=int((d > 0).sum()), clicks_worse=int((d < 0).sum()), clicks_same=int((d == 0).sum()), worst_click_drop=int(-d.min()), clicks_down_2plus=int((d <= -2).sum()), clicks_up_2plus=int((d >= 2).sum()), mean_worst_click_b=worst_b.mean(), mean_worst_click_a=worst_a.mean(), rois_worst_better=int((worst_a > worst_b).sum()), rois_worst_worse=int((worst_a < worst_b).sum()), rois_worst0_b=', '.join(f.replace('.tiff', '') for f in worst_b.index[worst_b == 0]), rois_worst0_a=', '.join(f.replace('.tiff', '') for f in worst_a.index[worst_a == 0]), mean_click_sd_b=w[b].groupby('file_name').std().mean(), mean_click_sd_a=w[a].groupby('file_name').std().mean()))
PER_CLICK = pd.DataFrame(pc_rows)
print('per click at tm_score, K = 10 (294 clicks; the worst click per ROI is the product\'s bad-seed case):')
print(PER_CLICK.T.to_string())
for b, a in (('A1', 'A3'),):
    d = (w[a] - w[b])
    print(f'\n{a} - {b}: clicks that lose (file, seed: {b} -> {a}):', ', '.join(f"{f.replace('.tiff', '')} s{s}: {int(w.loc[(f, s), b])}->{int(w.loc[(f, s), a])}" for (f, s) in d.index[d < 0]))
    zb, za = w[b] == 0, w[a] == 0
    print(f'{a} - {b}: zero-TP@10 clicks', f'{int(zb.sum())} -> {int(za.sum())};', 'new:', [f"{f.replace('.tiff', '')} s{s}" for f, s in w.index[za & ~zb]], 'rescued:', [f"{f.replace('.tiff', '')} s{s}" for f, s in w.index[zb & ~za]])


per click at tm_score, K = 10 (294 clicks; the worst click per ROI is the product's bad-seed case):
                                     0                   1                   2                   3
contrast                       A3 - A1             A2 - A0             A3 - A0             A1 - A0
clicks_better                       95                  96                 106                  17
clicks_worse                        20                  23                  21                   5
clicks_same                        179                 175                 167                 272
worst_click_drop                     1                   1                   1                   1
clicks_down_2plus                    0                   0                   0                   0
clicks_up_2plus                     19                  18                  21                   0
mean_worst_click_b            4.163265            4.061224            4.061224            4.061224
mean_wors

## 8. Robustness of the clip contrasts

Four checks:
- leave one ROI out;
- leave one domain out;
- each seed index separately (one click per ROI, so 49 clicks each);
- the three largest-gain ROIs removed.

In [12]:
rb_rows = []
for b, a in (('A1', 'A3'), ('A0', 'A2')):
    for k in BUDGETS:
        d = pd.Series(roi_sum(PER_RUN, PRODUCT_AXIS, a, f'tp_at_{k}') - roi_sum(PER_RUN, PRODUCT_AXIS, b, f'tp_at_{k}'), index=ROI_ORDER)
        full = d.sum() / (6 * k * N_ROIS)
        loo = {f: (d.sum() - d[f]) / (6 * k * (N_ROIS - 1)) for f in ROI_ORDER}
        lodo = {SHORT[dm]: d[[f for f in ROI_ORDER if DOMAIN_OF[f] != dm]].sum() / (6 * k * 42) for dm in DOMAINS}
        per_seed = {}
        for s in SEED_INDICES:
            sel = PER_RUN[(PER_RUN['rank_key'] == PRODUCT_AXIS) & (PER_RUN['seed_index'] == s)]
            per_seed[s] = (sel.loc[sel['arm'] == a, f'tp_at_{k}'].sum() - sel.loc[sel['arm'] == b, f'tp_at_{k}'].sum()) / (k * N_ROIS)
        top3 = d.sort_values(ascending=False).head(3)
        rb_rows.append(dict(contrast=f'{a} - {b}', K=k, pooled_pts=100 * full, loo_min_pts=100 * min(loo.values()), loo_min_roi=min(loo, key=loo.get), lodo_min_pts=100 * min(lodo.values()), lodo_min_domain=min(lodo, key=lodo.get), lodo_max_pts=100 * max(lodo.values()), per_seed_pts=', '.join(f'{100 * v:+.1f}' for v in per_seed.values()), seeds_positive=int(sum(v > 0 for v in per_seed.values())), top3_rois=', '.join(f"{f.replace('.tiff', '')} {int(v):+d}" for f, v in top3.items()), drop_top3_pts=100 * (d.sum() - top3.sum()) / (6 * k * (N_ROIS - 3))))
ROBUST = pd.DataFrame(rb_rows)
print('robustness of the clip contrasts at tm_score: leave one ROI out, leave one domain out, per seed index (each seed index is one click per ROI), and without the three largest ROIs:')
print(ROBUST.to_string(index=False, float_format=lambda v: f'{v:.2f}'))


robustness of the clip contrasts at tm_score: leave one ROI out, leave one domain out, per seed index (each seed index is one click per ROI), and without the three largest ROIs:
contrast  K  pooled_pts  loo_min_pts loo_min_roi  lodo_min_pts lodo_min_domain  lodo_max_pts                       per_seed_pts  seeds_positive                 top3_rois  drop_top3_pts
 A3 - A1 10        3.30         2.92    291.tiff          2.70   lymphosarcoma          3.61 +2.7, +3.1, +3.3, +2.9, +3.5, +4.5               6   291 +13, 220 +7, 284 +6           2.57
 A3 - A1 20        2.62         2.38    435.tiff          2.26   lymphosarcoma          2.84 +2.4, +3.5, +2.3, +2.9, +2.1, +2.4               6 435 +17, 220 +12, 291 +12           2.05
 A3 - A1 30        2.30         2.07    220.tiff          2.00            lung          2.58 +1.9, +2.4, +2.9, +1.8, +3.0, +1.9               6 220 +24, 435 +16, 291 +16           1.78
 A2 - A0 10        3.20         2.81    291.tiff          2.74   lymphosarcoma    

## 9. Verdicts per arm

Built from the tables above and written to `..._verdicts.csv`. The reading follows.

In [13]:
def row(axis, pair, k):
    """
        One recorded contrast row.

        axis (str): the rank_key.
        pair (str): the contrast name.
        k (int): the budget.

        Returns pd.Series: the DELTA row.
    """
    return DELTA[(DELTA['rank_key'] == axis) & (DELTA['pair'] == pair) & (DELTA['K'] == k)].iloc[0]


def fmt(r):
    """
        A contrast row as points with its bootstrap CI, W/L/T and Holm p.

        r (pd.Series): a DELTA row.

        Returns str: one line.
    """
    holm = '' if pd.isna(r['holm_p']) else f", Holm {r['holm_p']:.2g}"
    return f"{100 * r['pooled_delta_precision']:+.2f} [{100 * r['ci_low']:+.2f}, {100 * r['ci_high']:+.2f}], {r['wins']}/{r['losses']}/{r['ties']}{holm}"


wc = WHITE_CHECKS.set_index(['rank_key', 'contrast', 'white_cut'])
ch1 = CHURN.set_index(['arm_from', 'arm_to', 'tol_px'])
bd = BOUNDS.set_index(['rank_key', 'pair', 'K'])
VERDICTS = pd.DataFrame([
    dict(arm='A0', role='production mo_pad61, as recorded', verdict='baseline; superseded by the clip', evidence=f"P@10 tm {POOLED.loc['A0', ('P@10', 'tm_score')]:.3f}, od {POOLED.loc['A0', ('P@10', 'chromatin_od')]:.3f}; reproduces the recorded run exactly (source gate 3, audit Tier B)", still_open='-'),
    dict(arm='A1', role='band out (invalid + floored): fix 1', verdict='ADOPT as a benchmark-harness rule for every number on these ROI crops; not a production change', evidence=f"tm K=10 {fmt(row('tm_score', 'A1 - A0', 10))} (n.s.); od K=10 {fmt(row('chromatin_od', 'A1 - A0', 10))}; boundary-row detections 252 -> 0 (by construction)", still_open=f'adopting it breaks comparability with every recorded per_run reference (od jumps ~+2.3 pts at K=10; 1.0 % of annotated mitoses, those within 30 px of the edge, become unreachable), so earlier numbers need a re-run or a stated harness change; no product effect, since the product reads real pixels past a frame edge'),
    dict(arm='A1v', role='band out, valid-only (the prompt spec)', verdict='RETIRE: leaks band scores into the local-max filter; A1 is the correct implementation', evidence=f"A1 - A1v at tm K=10 {100 * row('tm_score', 'A1 - A1v', 10)['pooled_delta_precision']:+.2f} pts, largest difference allowed {bd.loc[('tm_score', 'A1 - A1v', 10), 'largest_diff_not_excluded_pts']:.2f} pts", still_open='-'),
    dict(arm='A2', role='floor clip at b_otsu: fix 2, the product change', verdict='ADOPT the clip as the production change (needs a dated D3 amendment); its product estimate is A3 - A1, the clip with the benchmark harness corrected on both sides', evidence=f"tm K=10 {fmt(row('tm_score', 'A2 - A0', 10))}; harness-corrected A3 - A1 {fmt(row('tm_score', 'A3 - A1', 10))}; od K=10 {fmt(row('chromatin_od', 'A2 - A0', 10))} (neutral)", still_open='held-out ROIs; rounds after the first; image-side vs template-side share of the gain'),
    dict(arm='A3', role='band out + clip at b_otsu: the pre-registered candidate', verdict='the pre-registered rule recommends it, and it stands; read it as A2 (the product change) measured on the corrected harness A1', evidence=f"primary tm K=10 {fmt(row('tm_score', 'A3 - A0', 10))}; of which band {100 * row('tm_score', 'A1 - A0', 10)['pooled_delta_precision']:+.2f}, clip {100 * row('tm_score', 'A3 - A1', 10)['pooled_delta_precision']:+.2f}", still_open='as A2'),
    dict(arm='A3v', role='A3 with the spec band', verdict='RETIRE with A1v', evidence=f"A3 - A3v at tm K=10 {100 * row('tm_score', 'A3 - A3v', 10)['pooled_delta_precision']:+.2f} pts, largest difference allowed {bd.loc[('tm_score', 'A3 - A3v', 10), 'largest_diff_not_excluded_pts']:.2f} pts", still_open='-'),
    dict(arm='A2m', role='floor clip at the ROI median', verdict='NOT ADOPTED, not worse: no difference from b_otsu is detected; b_otsu stays because it was pre-registered', evidence=f"A2m - A2 at tm K=10 {fmt(row('tm_score', 'A2m - A2', 10))}; largest difference allowed {bd.loc[('tm_score', 'A2m - A2', 10), 'largest_diff_not_excluded_pts']:.2f} pts (K=10), {bd.loc[('tm_score', 'A2m - A2', 20), 'largest_diff_not_excluded_pts']:.2f} (K=20)", still_open='whether a floor that clips less (a glass-only floor) keeps the gain; not tested'),
])
print(VERDICTS.to_string(index=False))

for name, frame in (('contrasts', CONTRAST_TAB), ('white_checks', WHITE_CHECKS), ('bounds', BOUNDS), ('churn_top10', CHURN), ('white_dose', DOSE), ('white_dose_gap', DOSE_GAP), ('pool_od_flows', FLOWS), ('domain', DOMAIN_TAB), ('domain_level', DOMAIN_LEVEL), ('per_click', PER_CLICK), ('robustness', ROBUST), ('verdicts', VERDICTS)):
    frame.to_csv(OUT[name], index=False)
print('\nwritten:', *OUT.values(), sep='\n  ')


arm                                                    role                                                                                                                                                            verdict                                                                                                                                                                            evidence                                                                                                                                                                                                                                                                                                                 still_open
 A0                        production mo_pad61, as recorded                                                                                                                                   baseline; superseded by the clip                                                            

## Reading

Every number is in-sample on the 49 ROIs and post hoc, and every gate above held.

### Verdict per arm

| arm | verdict | the evidence it rests on |
|---|---|---|
| `A0` | **Baseline, superseded by the clip.** | P@10 at `tm_score` 0.542. It reproduces the recorded `mo_pad61` run exactly (source gate 3, audit Tier B). |
| `A1` | **Adopt as a benchmark-harness rule** for every future number on these ROI crops. It is not a production change. **Cost:** it breaks comparability with every recorded `per_run` reference, including the post-D11 `tm_score` reference and the regression reference. Earlier numbers need a re-run or a stated harness change. | At `tm_score`, +0.41 pts at K = 10, Holm 0.25 (n.s.). At chromatin od, +2.28 at K = 10, Holm 3e-4, because od's 51 px window is replicate-padded too, so od numbers jump on adoption. 28 of 2782 annotated mitoses (1.0 %) lie within 30 px of an edge and become unreachable. The product reads real pixels past any frame edge, so this fix has nothing to remove there. |
| `A1v` | **Retire.** `A1` is the correct implementation of fix 1. | The spec version leaks band scores into the local-max filter. The top 10s are identical on 287 / 294 clicks, and the CI allows at most 0.10 pt at K = 10. |
| `A2` | **Adopt the clip as the production change.** It needs a dated amendment to D3 (below). | Product estimate `A3 - A1`: **+3.30 pts at K = 10** [+2.11, +4.63], 32 / 6 / 11 ROIs, Holm 8e-6. +2.62 at K = 20 and +2.30 at K = 30. D5 bar met at every K. Neutral on chromatin od. |
| `A3` | **The pre-registered recommendation stands.** Read it as `A2` measured on the corrected harness `A1`. | +3.71 = band +0.41 + clip +3.30, and the two add. |
| `A3v` | **Retire**, with `A1v`. | Identical top 10s on 287 / 294 clicks; the CI allows at most 0.17 pt at K = 10. |
| `A2m` | **Not adopted, and not worse.** `b_otsu` stays because it was pre-registered. | `A2m - A2` is +0.03 pts at K = 10 [-0.44, +0.51]. At `tm_score`, the two intervals together allow at most 0.54 / 0.58 / 0.24 pt at K = 10 / 20 / 30; the widest bound on either axis is 0.70 pt (od, K = 20). These are descriptive bounds with no pre-set margin. |

**Why `A2m` matters despite tying.** `A2m` and `A2` reach the same precision with different lists. Their top 10s are
identical on only 33 / 294 clicks. `b_med` clips 48-50 % of every ROI, while `b_otsu` clips 27-83 %.
So the precision gain does not hinge on the exact floor level within that range.

### The clip on the product contrast (`A3 - A1`, `tm_score`)

| check | result |
|---|---|
| pooled, K = 10 / 20 / 30 | +3.30 / +2.62 / +2.30 pts; Holm 8e-6 / 5e-7 / 2e-6; D5 bar met at every K |
| domain as the unit | positive in 7 / 7 domains at every K. Exact p 1/64, Holm over K 0.047. That is the attainable floor, so this survives a domain-level null only by unanimity. |
| leave one ROI out, K = 10 | minimum +2.92 (without `291.tiff`) |
| leave one domain out, K = 10 | +2.70 (without lymphosarcoma) to +3.61 |
| three largest ROIs removed, K = 10 | +2.57 |
| each seed index alone, K = 10 | positive in 6 / 6: +2.7, +3.1, +3.3, +2.9, +3.5, +4.5 |
| per click, K = 10 | 95 better, 20 worse, 179 the same. Every loss is exactly 1 TP; 19 clicks gain 2 or more. |
| worst click per ROI | mean TP@10 4.16 -> 4.55. Better on 18 ROIs, worse on 3. |
| zero-TP@10 clicks | 13 -> 11. Rescued: `242` s1, `242` s3, `462` s0. New: `245` s1. |
| the generator | pool mitoses by annotation id: 611 enter and 163 leave, net +448 (+1.5 per click) |
| chromatin od | -0.48 pts at K = 10 [-1.29, +0.24], n.s.; 2 / 5 domains. The clip is neutral on the od axis. |

### What the clip does, corrected

At the detection level the clip does what the white-surround mechanism predicts. The source notebook's
"not a targeted removal" is overturned. Its ROI-level observation survives, for a reason given below.

**Targeting, on the product contrast.**
- **M2 falls.** The white-adjacent share of top-10 false positives goes 0.237 -> 0.194, CI [-0.060, -0.027].
- **White-adjacent FPs fall in every ROI that changes.** In counts they go 316 -> 240 (-76). No ROI gains any, 27
  lose, 22 tie; exact p 1.5e-8.
- **The other FPs barely move.** They go 1019 -> 998 (-21; 17 / 20 / 12 ROIs, p 0.24).
- **The cut-off does not drive this.**
  - At 10 % white: white-adjacent -105, others +8.
  - At 30 %: white-adjacent -40, others -57 (p 0.002). The 20-30 % candidates now sit on the "other" side.

**Churn counted by object (1 px identity).**
- 438 of 2940 top-10 slots change, not 1164.
- 150 mitoses leave and 247 enter; by annotation id, 122 and 219.
- Of the FPs that leave, 32.6 % are white-adjacent, against a 23.7 % base rate. Of those that enter, 9.4 % are.

**Dose-response** (leave rate from the base top 10).
- False positives sit slightly lower in the base top 10 than mitoses (mean rank 4.80 against 4.25), and a candidate
  near rank 10 leaves far more often than one near rank 1.
- So the gap is also given rank-standardised: the mitosis rate is taken per rank stratum (0-4, 5-7, 8-9) and weighted
  by the false positives' strata.

| white share of the 61 px window | FPs that leave | mitoses that leave | FP minus mitosis, raw | rank-standardised |
|---|---|---|---|---|
| <= 2 % | 13.6 % | 8.6 % | +5.1 [+0.4, +9.3] | +3.6 [-1.7, +8.3] |
| 2-10 % | 22.7 % | 10.3 % | +12.5 [+6.3, +18.4] | +11.1 [+5.5, +16.7] |
| 10-20 % | 20.8 % | 10.3 % | +10.5 [+3.6, +17.3] | +10.6 [+3.8, +16.8] |
| 20-40 % | 30.4 % | 9.7 % | +20.7 [+11.6, +30.7] | +16.4 [+6.9, +27.7] |
| > 40 % | 27.8 % | 2.7 % (n = 37) | +25.1 [+9.1, +34.3] | +21.1 [+4.5, +33.3] |

In the > 40 % bin, 8,423 of the 10,000 resamples are usable, because the standardisation needs a mitosis in every rank
stratum.

Three readings follow.
- **The clip demotes false positives more the whiter their surround is.**
  - Mitoses leave at a flat 9-10 % whatever their surround, so the clip is not a blind surround penalty.
  - The gradient survives within rank strata. At ranks 8-9, false positives leave at 31 % with <= 2 % white and at
    54-63 % with more; mitoses leave at 25-33 %.
  - No rise in the leave rate of white-adjacent mitoses is detected. The > 40 % bin is small (n = 37) and cannot rule
    one out.
  - M3 on the product contrast is -6 white-adjacent mitoses in the top 30 (p 0.34, CI [-16, +3]).
- **The vacated slots fill from dense tissue.**
  - 80 of 191 entering FPs and 124 of 247 entering mitoses sit in <= 2 % white.
  - The net mitosis gain is +67 in the <= 2 % bin and +34 in the 2-10 % bin.
  - So the gain is mitoses in plain tissue that white-surrounded false positives used to outrank.
- **No discrimination is detected without white.** In essentially white-free surrounds (<= 2 %), the raw +5.1 pt gap
  shrinks to +3.6 [-1.7, +8.3] once rank is accounted for. The source notebook's "the floor also flattens cytoplasm and
  stroma" is therefore not needed to explain the gain, and this data cannot detect it. Note that even 2-10 % white, a
  small patch of lumen or glass in the 61 px window, is enough for the clip to act.

**Why ROI white share still fails to predict the gain** (Spearman +0.08, source notebook). False positives cluster at
the little white a ROI has. Lymphosarcoma is 3.9 % white by area, but 24 % of its top-10 false positives are
white-adjacent.

**Mast cell tumour is the real exception.** The clip removes only 9 % of its white-adjacent false positives (11 of
121), and its other false positives rise by 4. Its gain (+1.7) is among the smallest.

### Pool and chromatin od, corrected (`A1 -> A3`)

**Pool.** By annotation id, 163 mitoses leave the pool and 611 enter. The source notebook's ~2,520 / ~2,960 counted
pixel shifts.

**Od top 10, mitoses.** 58 enter, and 55 of them are new to the pool. 72 leave, and only 4 of those are gone from the
pool. New-to-pool mitoses do reach od's top 10, so the source notebook's "they do not reach it on net" is wrong.

**Od top 10, non-mitoses.** 134 enter, and 126 of them are new to the pool. The clip pools dark candidates that od
ranks high: non-white FPs in od's top 10 rise by 25 (p 0.027), while white-adjacent FPs fall by 11. The net is -14 TP.
That is exactly "od sees the pool set". The clip's pool set gains mitoses and high-od non-mitoses alike.

### D3

The clip is a bottom clip. D3's "what would change my mind" clause admits only transforms that "do not clip", so
adoption needs a dated amendment. Note for that amendment: `hematoxylin_od` already carries a bottom clip at 0, from
`rgb2hed`'s clamp. A median of 3.9 % of pixels per ROI are exactly 0 (audit Tier B).

### Caveats that stay attached to the verdict

- **In-sample.** These are 49 ROIs, the hypothesis was formed on them, and D5 calls the draw optimistic. There is no
  held-out confirmation.
- **Round 1 only.** Later rounds face a depleted pool with prior candidates blanked.
- **The domain-level result sits exactly at its attainable floor** (7 / 7).
- **M3's lower bound** is -16 white-adjacent mitoses in the top 30 on the product contrast, and -24 on `A3 - A0`. A
  small loss is not excluded.
- **The white proxy** (gray > 215 in a 61 px window) was never validated against tissue classes.
- **The floor is computed per benchmark ROI.** In the product, the region it is computed over (frame, viewport or
  slide) is not defined yet. `A2m` = `A2` suggests the level is forgiving, but regions of other sizes were not tested.

### Open, for the user to decide

1. **Held-out confirmation** of `A3 - A1` on unused MIDOG++ ROIs. It was declined on 2026-09-30. 6 domains are
   possible; neuroendocrine has no valid spare ROI.
2. **The image-side vs template-side split of the clip.** One arm clips the image but cuts the template from `hem`; the
   other does the reverse. This explains the mechanism but does not change adoption, because `A2` as tested is the
   unit that would ship.
3. **The D3 amendment and the production implementation** of the clip, including how `b` is computed in the product.
